<a href="https://colab.research.google.com/github/elkins-lab/synth-saxs/blob/main/examples/interactive_tutorials/hiv1_rt_hinge_motion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Detecting Allosteric Hinge Motions: HIV-1 Reverse Transcriptase

Inspired by the structural biology research of the Arnold Lab (Rutgers University), this tutorial demonstrates how to use `synth-saxs` to detect large-scale conformational changes in solution.

HIV-1 Reverse Transcriptase (RT) is a highly flexible enzyme. Its "thumb" domain acts like a hinge. The Arnold Lab's pioneering work revealed that while it transitions from an **"open"** unliganded state to a **"closed"** state when bound to a DNA/RNA duplex, Non-Nucleoside Reverse Transcriptase Inhibitors (NNRTIs) actually wedge the hinge into a **"hyperextended open"** state, locking it and preventing DNA polymerization.

We will compute the SAXS profile and $P(r)$ distance distribution for all three states (Open, Closed, and Hyperextended) to see how this hinge motion manifests in the scattering data.

In [ ]:
import sys

if "google.colab" in sys.modules:
    %pip install -q git+https://github.com/elkins-lab/synth-saxs.git biotite matplotlib py3Dmol
else:
    sys.path.append("../../")

In [ ]:
import biotite.database.rcsb as rcsb
import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import numpy as np
import py3Dmol

from synth_saxs import calculate_p_dist, calculate_saxs_profile, preprocess_structure

## 1. Fetching the Conformational States
We will download three distinct structures of HIV-1 RT from the PDB:
- **1DLO**: Unliganded, "Open" state (Arnold Lab).
- **1RTD**: DNA-bound, "Closed" state.
- **1S1X**: NNRTI-bound (Etravirine), "Hyperextended Open" state (Arnold Lab).

In [ ]:
print("Fetching structures from RCSB PDB...")

# Fetch Open State
file_open = rcsb.fetch("1DLO", "pdb", ".")
struct_open = strucio.load_structure(file_open)

# Fetch Closed State (DNA-bound)
file_closed = rcsb.fetch("1RTD", "pdb", ".")
struct_closed = strucio.load_structure(file_closed)

# Fetch NNRTI-bound State
file_nnrti = rcsb.fetch("1S1X", "pdb", ".")
struct_nnrti = strucio.load_structure(file_nnrti)

print("Structures loaded successfully.")

## 2. Preprocessing & Preserving Nucleic Acids
Crystal structures often contain water and crystallization buffer which must be removed for SAXS simulation. We use `preprocess_structure` to strip the buffer while safely preserving the nucleic acids.

**Note on 1RTD**: The PDB file for 1RTD actually contains two full complexes in the asymmetric unit! To accurately compare a single heterodimer across states, we filter 1RTD to just chains A, B, E, and F.

To significantly reduce memory requirements and compute time, we coarse-grain the structure to just its backbone atoms (`CA` for proteins, `P` for DNA).

In [ ]:
# Preprocess Open State
clean_open = preprocess_structure(struct_open, keep_nucleic_acids=True)
clean_open = clean_open[(clean_open.atom_name == "CA") | (clean_open.atom_name == "P")]

# Preprocess Closed State (Filter out 2nd asymmetric complex!)
clean_closed = preprocess_structure(struct_closed, keep_nucleic_acids=True)
clean_closed = clean_closed[np.isin(clean_closed.chain_id, ["A", "B", "E", "F"])]
clean_closed = clean_closed[(clean_closed.atom_name == "CA") | (clean_closed.atom_name == "P")]

# Preprocess NNRTI-bound State
clean_nnrti = preprocess_structure(struct_nnrti, keep_nucleic_acids=True)
clean_nnrti = clean_nnrti[(clean_nnrti.atom_name == "CA") | (clean_nnrti.atom_name == "P")]

print(f"Open state: {len(clean_open)} atoms.")
print(f"Closed state: {len(clean_closed)} atoms.")
print(f"Hyperextended state: {len(clean_nnrti)} atoms.")

## 3. Interactive 3D Visualization
Using `py3Dmol`, let's visualize the immense flexibility of the p66 thumb subdomain. 
We will load the three structures and color them to distinguish the states:
- **Blue**: Open (Unliganded)
- **Red**: Closed (DNA-bound)
- **Green**: Hyperextended (NNRTI-bound)

*Note: The structures in the PDB are not aligned in space. For a true visual overlay, they would need to be superimposed on the p51 rigid body.*

In [ ]:
view = py3Dmol.view(width=800, height=500)

with open(file_open) as f:
    view.addModel(f.read(), "pdb")
with open(file_closed) as f:
    view.addModel(f.read(), "pdb")
with open(file_nnrti) as f:
    view.addModel(f.read(), "pdb")

# Open = Blue, Closed = Red, Hyperextended = Green
view.setStyle({"model": 0}, {"cartoon": {"color": "blue"}})
view.setStyle({"model": 1, "chain": ["A", "B"]}, {"cartoon": {"color": "red"}})
view.setStyle({"model": 1, "chain": ["E", "F"]}, {"cartoon": {"color": "orange"}})
view.setStyle({"model": 2}, {"cartoon": {"color": "green"}})

view.zoomTo({"model": 0})
view.show()

## 4. Simulating the SAXS Profiles
We calculate the $I(q)$ curves for all three states using the standard Debye formulation.

In [ ]:
print("Simulating SAXS profile for Open State...")
q_open, i_open = calculate_saxs_profile(clean_open, q_max=0.3, n_points=150)

print("Simulating SAXS profile for Closed State...")
q_closed, i_closed = calculate_saxs_profile(clean_closed, q_max=0.3, n_points=150)

print("Simulating SAXS profile for Hyperextended State...")
q_nnrti, i_nnrti = calculate_saxs_profile(clean_nnrti, q_max=0.3, n_points=150)

## 5. Distance Distribution Analysis ($P(r)$)
The Pair-Distance Distribution $P(r)$ is highly sensitive to hinge motions. 
- As the thumb closes (DNA bound), the maximum dimension ($D_{max}$) shrinks.
- As the NNRTI wedges the hinge open, the $D_{max}$ actually expands beyond the unliganded open state!

In [ ]:
print("Calculating P(r) distributions...")
r_open, p_open = calculate_p_dist(clean_open, bins=100)
r_closed, p_closed = calculate_p_dist(clean_closed, bins=100)
r_nnrti, p_nnrti = calculate_p_dist(clean_nnrti, bins=100)

# Normalize P(r) for direct shape comparison
p_open /= np.max(p_open)
p_closed /= np.max(p_closed)
p_nnrti /= np.max(p_nnrti)

plt.figure(figsize=(8, 5))
plt.plot(r_open, p_open, label="Open (1DLO)", color="blue", linewidth=2)
plt.plot(r_closed, p_closed, label="Closed (1RTD)", color="red", linewidth=2, linestyle="--")
plt.plot(r_nnrti, p_nnrti, label="Hyperextended (1S1X)", color="green", linewidth=2, linestyle=":")

plt.title("Pair-Distance Distribution P(r) of HIV-1 RT")
plt.xlabel(r"Distance $r$ ($\AA$)")
plt.ylabel("Normalized P(r)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(
    "Notice how the Closed state shifts inward, while the Hyperextended state pushes the tail outward!"
)

## 6. Difference Curve Analysis ($\Delta I/I$)
A difference curve directly compares the scattering intensities of two states against a reference state. It is a highly sensitive and standard way to visualize structural changes. The oscillating "waves" in a difference curve act as a unique fingerprint for specific domain movements!

In [ ]:
print("Calculating Difference Curves (Relative to Open State)...")
percent_diff_closed = ((i_closed - i_open) / i_open) * 100
percent_diff_nnrti = ((i_nnrti - i_open) / i_open) * 100

plt.figure(figsize=(8, 5))
plt.plot(q_open, percent_diff_closed, color="red", linewidth=2, label="Closed vs Open")
plt.plot(q_open, percent_diff_nnrti, color="green", linewidth=2, label="Hyperextended vs Open")
plt.axhline(0, color="black", linestyle="--", alpha=0.5)

plt.title("SAXS Difference Curves")
plt.xlabel(r"$q$ ($\AA^{-1}$)")
plt.ylabel(r"$\Delta I / I_{open}$ (%)")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(
    "The large, distinct oscillations are classic signatures of large-scale domain rearrangements!"
)